# 03 — PPO Training

Train a **PPO** (Proximal Policy Optimization) agent on the *Deadly Corridor*
and *Defend the Center* scenarios. Compare sample efficiency vs DQN and
visualize policy entropy and value loss over time.

## 1. Setup

In [ ]:
# Uncomment on Colab
# !pip install vizdoom gymnasium torch opencv-python matplotlib tensorboard

import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
import torch

from rl_doom.env import DoomEnv, ResizeObservation, SkipFrame, FrameStack
from rl_doom.models import ActorCriticNetwork
from rl_doom.agents.ppo import PPOAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["checkpoints", "logs", "figures", "media", "runs"]:
#     drive_dir = f"{DRIVE_ROOT}/{subdir}"
#     local_dir = os.path.abspath(f"../{subdir}")
#     os.makedirs(drive_dir, exist_ok=True)
#     if os.path.islink(local_dir):
#         os.remove(local_dir)
#     if os.path.isdir(local_dir):
#         for f in os.listdir(local_dir):
#             src = os.path.join(local_dir, f)
#             dst = os.path.join(drive_dir, f)
#             if not os.path.exists(dst):
#                 shutil.move(src, dst)
#         shutil.rmtree(local_dir)
#     os.symlink(drive_dir, local_dir)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

## 2. Environment factory

In [ ]:
def make_env(scenario="deadly_corridor", seed=None):
    env = DoomEnv(scenario=scenario)
    env = ResizeObservation(env, shape=(84, 84))
    env = SkipFrame(env, skip=4)
    env = FrameStack(env, num_stack=4)
    return env

env = make_env("deadly_corridor")
obs, _ = env.reset(seed=42)
n_actions = env.action_space.n
print(f"Obs shape: {obs.shape}, Actions: {n_actions}")

## 3. PPO Hyperparameters

In [ ]:
config = dict(
    # Rollout
    n_steps=2048,           # steps per rollout
    n_epochs=4,             # PPO epochs per update
    batch_size=64,          # mini-batch size
    total_timesteps=200_000,
    # PPO
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    # Logging
    log_freq=1,             # log every N updates
    eval_freq=5,            # evaluate every N updates
    eval_episodes=10,
)
config

## 4. Initialize agent

In [ ]:
agent = PPOAgent(
    obs_shape=obs.shape,
    n_actions=n_actions,
    lr=config["lr"],
    gamma=config["gamma"],
    gae_lambda=config["gae_lambda"],
    clip_eps=config["clip_eps"],
    entropy_coef=config["entropy_coef"],
    value_coef=config["value_coef"],
    max_grad_norm=config["max_grad_norm"],
    device=device,
)

print(f"Parameters: {sum(p.numel() for p in agent.network.parameters()):,}")

## 5. Rollout collection helper

In [ ]:
def collect_rollout(env, agent, n_steps, obs=None):
    """Collect a single rollout of n_steps transitions.

    Args:
        env: The environment to collect from.
        agent: The PPO agent.
        n_steps: Number of steps to collect.
        obs: Starting observation. If None, resets the environment.

    Returns:
        rollout: Dict of collected transitions.
        episode_rewards: List of completed episode rewards.
        episode_lengths: List of completed episode lengths.
        last_obs: The observation after the last step (to carry over to next rollout).
    """
    obs_buf = []
    act_buf = []
    rew_buf = []
    done_buf = []
    logp_buf = []
    val_buf = []

    if obs is None:
        obs, _ = env.reset()
    episode_rewards = []
    episode_lengths = []
    ep_reward = 0.0
    ep_steps = 0

    for _ in range(n_steps):
        action, log_prob, value = agent.select_action(obs)

        obs_buf.append(obs)
        act_buf.append(action)
        logp_buf.append(log_prob)
        val_buf.append(value)

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        rew_buf.append(reward)
        done_buf.append(done)

        ep_reward += reward
        ep_steps += 1
        if done:
            episode_rewards.append(ep_reward)
            episode_lengths.append(ep_steps)
            ep_reward = 0.0
            ep_steps = 0
            obs, _ = env.reset()
        else:
            obs = next_obs

    # Bootstrap value for last state
    _, _, last_value = agent.select_action(obs)

    rollout = dict(
        obs=np.array(obs_buf),
        actions=np.array(act_buf),
        rewards=np.array(rew_buf),
        dones=np.array(done_buf),
        log_probs=np.array(logp_buf),
        values=np.array(val_buf),
        last_value=last_value,
    )
    return rollout, episode_rewards, episode_lengths, obs

## 6. Training — Deadly Corridor

In [ ]:
import time
from torch.utils.tensorboard import SummaryWriter

os.makedirs("../runs", exist_ok=True)
writer_dc = SummaryWriter(log_dir="../runs/ppo_deadly_corridor")

n_updates = config["total_timesteps"] // config["n_steps"]
print(f"Total updates: {n_updates}")

all_episode_rewards = []
all_episode_lengths = []
policy_losses = []
value_losses = []
entropies = []
clip_fractions = []
eval_log = []

carry_obs = None
t_start = time.time()
total_env_steps = 0

for update in range(1, n_updates + 1):
    rollout, ep_rews, ep_lens, carry_obs = collect_rollout(
        env, agent, config["n_steps"], obs=carry_obs
    )
    all_episode_rewards.extend(ep_rews)
    all_episode_lengths.extend(ep_lens)
    total_env_steps += config["n_steps"]

    stats = agent.update(
        rollout,
        n_epochs=config["n_epochs"],
        batch_size=config["batch_size"],
    )
    policy_losses.append(stats["policy_loss"])
    value_losses.append(stats["value_loss"])
    entropies.append(stats["entropy"])
    clip_frac = stats.get("clip_fraction", 0.0)
    clip_fractions.append(clip_frac)

    # TensorBoard logging
    writer_dc.add_scalar("train/policy_loss", stats["policy_loss"], update)
    writer_dc.add_scalar("train/value_loss", stats["value_loss"], update)
    writer_dc.add_scalar("train/entropy", stats["entropy"], update)
    writer_dc.add_scalar("train/clip_fraction", clip_frac, update)
    for r in ep_rews:
        writer_dc.add_scalar("episode/reward", r, len(all_episode_rewards))
    for l in ep_lens:
        writer_dc.add_scalar("episode/length", l, len(all_episode_lengths))

    if update % config["log_freq"] == 0:
        recent = all_episode_rewards[-20:] if all_episode_rewards else [0]
        elapsed = time.time() - t_start
        fps = total_env_steps / elapsed if elapsed > 0 else 0
        print(
            f"Update {update}/{n_updates} | "
            f"Episodes {len(all_episode_rewards)} | "
            f"Avg reward (20): {np.mean(recent):.2f} | "
            f"Entropy: {stats['entropy']:.4f} | "
            f"Clip: {clip_frac:.3f} | "
            f"FPS: {fps:.0f}"
        )

    if update % config["eval_freq"] == 0:
        eval_env = make_env("deadly_corridor")
        eval_rews = []
        eval_lens = []
        for _ in range(config["eval_episodes"]):
            eo, _ = eval_env.reset()
            er, el = 0.0, 0
            d = False
            while not d:
                a, _, _ = agent.select_action(eo)
                eo, r, term, trunc, _ = eval_env.step(a)
                er += r
                el += 1
                d = term or trunc
            eval_rews.append(er)
            eval_lens.append(el)
        eval_env.close()
        step_count = update * config["n_steps"]
        eval_log.append((step_count, np.mean(eval_rews), np.std(eval_rews),
                         np.mean(eval_lens), np.std(eval_lens)))
        writer_dc.add_scalar("eval/mean_reward", np.mean(eval_rews), step_count)
        writer_dc.add_scalar("eval/mean_length", np.mean(eval_lens), step_count)
        print(f"  >> Eval: {np.mean(eval_rews):.2f} +/- {np.std(eval_rews):.2f}")

wall_time_dc = time.time() - t_start
env.close()
writer_dc.close()
print(f"\nTraining complete — Deadly Corridor | Wall time: {wall_time_dc:.1f}s | FPS: {total_env_steps / wall_time_dc:.0f}")

## 7. Plot: Rewards, policy entropy, value loss

In [ ]:
import platform, datetime

os.makedirs("../figures", exist_ok=True)
os.makedirs("../logs", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Episode rewards
ax = axes[0, 0]
ax.plot(all_episode_rewards, alpha=0.3, label="Raw")
if len(all_episode_rewards) >= 20:
    sm = np.convolve(all_episode_rewards, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(sm)), sm, label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.set_title("Episode Rewards — Deadly Corridor")
ax.legend()

# Episode lengths
ax = axes[0, 1]
ax.plot(all_episode_lengths, alpha=0.3, color="green", label="Raw")
if len(all_episode_lengths) >= 20:
    sm = np.convolve(all_episode_lengths, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(sm)), sm, color="darkgreen", label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps")
ax.set_title("Episode Lengths")
ax.legend()

# Policy loss
ax = axes[0, 2]
ax.plot(policy_losses)
ax.set_xlabel("Update")
ax.set_ylabel("Policy Loss")
ax.set_title("PPO Policy Loss")

# Value loss
ax = axes[1, 0]
ax.plot(value_losses)
ax.set_xlabel("Update")
ax.set_ylabel("Value Loss")
ax.set_title("Value Function Loss")

# Entropy
ax = axes[1, 1]
ax.plot(entropies)
ax.set_xlabel("Update")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")

# Clip fraction
ax = axes[1, 2]
ax.plot(clip_fractions)
ax.set_xlabel("Update")
ax.set_ylabel("Clip Fraction")
ax.set_title("PPO Clip Fraction")

plt.tight_layout()
plt.savefig("../figures/03_ppo_deadly_corridor.png", dpi=150, bbox_inches="tight")
plt.show()

# Save training logs with reproducibility metadata
np.savez(
    "../logs/ppo_deadly_corridor_training.npz",
    episode_rewards=np.array(all_episode_rewards),
    episode_lengths=np.array(all_episode_lengths),
    policy_losses=np.array(policy_losses),
    value_losses=np.array(value_losses),
    entropies=np.array(entropies),
    clip_fractions=np.array(clip_fractions),
    eval_log=np.array(eval_log) if eval_log else np.array([]),
    # Reproducibility metadata
    config=str(config),
    seed=42,
    wall_time_seconds=wall_time_dc,
    fps=total_env_steps / wall_time_dc if wall_time_dc > 0 else 0,
    total_env_steps=total_env_steps,
    timestamp=str(datetime.datetime.now(datetime.timezone.utc)),
    python_version=platform.python_version(),
    platform_info=platform.platform(),
    torch_version=torch.__version__,
    device=str(device),
)

## 8. Train on Defend the Center

In [ ]:
writer_dtc = SummaryWriter(log_dir="../runs/ppo_defend_the_center")

env_dtc = make_env("defend_the_center")
obs_dtc, _ = env_dtc.reset(seed=42)
n_actions_dtc = env_dtc.action_space.n

agent_dtc = PPOAgent(
    obs_shape=obs_dtc.shape,
    n_actions=n_actions_dtc,
    lr=config["lr"],
    gamma=config["gamma"],
    gae_lambda=config["gae_lambda"],
    clip_eps=config["clip_eps"],
    entropy_coef=config["entropy_coef"],
    value_coef=config["value_coef"],
    max_grad_norm=config["max_grad_norm"],
    device=device,
)

dtc_rewards = []
dtc_lengths = []
dtc_policy_losses = []
dtc_value_losses = []
dtc_entropies = []
dtc_clip_fractions = []
dtc_eval_log = []
dtc_obs = None
t_start_dtc = time.time()
dtc_total_steps = 0

for update in range(1, n_updates + 1):
    rollout, ep_rews, ep_lens, dtc_obs = collect_rollout(
        env_dtc, agent_dtc, config["n_steps"], obs=dtc_obs
    )
    dtc_rewards.extend(ep_rews)
    dtc_lengths.extend(ep_lens)
    dtc_total_steps += config["n_steps"]

    stats = agent_dtc.update(rollout, n_epochs=config["n_epochs"], batch_size=config["batch_size"])
    dtc_policy_losses.append(stats["policy_loss"])
    dtc_value_losses.append(stats["value_loss"])
    dtc_entropies.append(stats["entropy"])
    clip_frac = stats.get("clip_fraction", 0.0)
    dtc_clip_fractions.append(clip_frac)

    # TensorBoard
    writer_dtc.add_scalar("train/policy_loss", stats["policy_loss"], update)
    writer_dtc.add_scalar("train/value_loss", stats["value_loss"], update)
    writer_dtc.add_scalar("train/entropy", stats["entropy"], update)
    writer_dtc.add_scalar("train/clip_fraction", clip_frac, update)
    for r in ep_rews:
        writer_dtc.add_scalar("episode/reward", r, len(dtc_rewards))
    for l in ep_lens:
        writer_dtc.add_scalar("episode/length", l, len(dtc_lengths))

    if update % config["log_freq"] == 0:
        recent = dtc_rewards[-20:] if dtc_rewards else [0]
        elapsed = time.time() - t_start_dtc
        fps = dtc_total_steps / elapsed if elapsed > 0 else 0
        print(
            f"Update {update}/{n_updates} | "
            f"Avg reward (20): {np.mean(recent):.2f} | "
            f"Entropy: {stats['entropy']:.4f} | "
            f"Clip: {clip_frac:.3f} | "
            f"FPS: {fps:.0f}"
        )

    if update % config["eval_freq"] == 0:
        eval_env_dtc = make_env("defend_the_center")
        eval_rews = []
        eval_lens = []
        for _ in range(config["eval_episodes"]):
            eo, _ = eval_env_dtc.reset()
            er, el = 0.0, 0
            d = False
            while not d:
                a, _, _ = agent_dtc.select_action(eo)
                eo, r, term, trunc, _ = eval_env_dtc.step(a)
                er += r
                el += 1
                d = term or trunc
            eval_rews.append(er)
            eval_lens.append(el)
        eval_env_dtc.close()
        step_count = update * config["n_steps"]
        dtc_eval_log.append((step_count, np.mean(eval_rews), np.std(eval_rews),
                             np.mean(eval_lens), np.std(eval_lens)))
        writer_dtc.add_scalar("eval/mean_reward", np.mean(eval_rews), step_count)
        writer_dtc.add_scalar("eval/mean_length", np.mean(eval_lens), step_count)
        print(f"  >> Eval: {np.mean(eval_rews):.2f} +/- {np.std(eval_rews):.2f}")

wall_time_dtc = time.time() - t_start_dtc
env_dtc.close()
writer_dtc.close()
print(f"\nTraining complete — Defend the Center | Wall time: {wall_time_dtc:.1f}s | FPS: {dtc_total_steps / wall_time_dtc:.0f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Episode rewards
ax = axes[0, 0]
ax.plot(dtc_rewards, alpha=0.3, label="Raw")
if len(dtc_rewards) >= 20:
    sm = np.convolve(dtc_rewards, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(sm)), sm, label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.set_title("Episode Rewards — Defend the Center")
ax.legend()

# Episode lengths
ax = axes[0, 1]
ax.plot(dtc_lengths, alpha=0.3, color="green", label="Raw")
if len(dtc_lengths) >= 20:
    sm = np.convolve(dtc_lengths, np.ones(20)/20, mode="valid")
    ax.plot(range(19, 19 + len(sm)), sm, color="darkgreen", label="MA-20")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps")
ax.set_title("Episode Lengths")
ax.legend()

# Policy loss
ax = axes[0, 2]
ax.plot(dtc_policy_losses)
ax.set_xlabel("Update")
ax.set_ylabel("Policy Loss")
ax.set_title("PPO Policy Loss")

# Value loss
ax = axes[1, 0]
ax.plot(dtc_value_losses)
ax.set_xlabel("Update")
ax.set_ylabel("Value Loss")
ax.set_title("Value Function Loss")

# Entropy
ax = axes[1, 1]
ax.plot(dtc_entropies)
ax.set_xlabel("Update")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")

# Clip fraction
ax = axes[1, 2]
ax.plot(dtc_clip_fractions)
ax.set_xlabel("Update")
ax.set_ylabel("Clip Fraction")
ax.set_title("PPO Clip Fraction")

plt.tight_layout()
plt.savefig("../figures/03_ppo_defend_the_center.png", dpi=150, bbox_inches="tight")
plt.show()

# Save training logs — now symmetric with Deadly Corridor
np.savez(
    "../logs/ppo_defend_the_center_training.npz",
    episode_rewards=np.array(dtc_rewards),
    episode_lengths=np.array(dtc_lengths),
    policy_losses=np.array(dtc_policy_losses),
    value_losses=np.array(dtc_value_losses),
    entropies=np.array(dtc_entropies),
    clip_fractions=np.array(dtc_clip_fractions),
    eval_log=np.array(dtc_eval_log) if dtc_eval_log else np.array([]),
    # Reproducibility metadata
    config=str(config),
    seed=42,
    wall_time_seconds=wall_time_dtc,
    fps=dtc_total_steps / wall_time_dtc if wall_time_dtc > 0 else 0,
    total_env_steps=dtc_total_steps,
    timestamp=str(datetime.datetime.now(datetime.timezone.utc)),
    python_version=platform.python_version(),
    platform_info=platform.platform(),
    torch_version=torch.__version__,
    device=str(device),
)

## 9. Sample efficiency: PPO vs DQN

Compare training curves from Notebook 02 (DQN) with the PPO curves above.
Load saved DQN results if available.

In [ ]:
# Placeholder — populate dqn_rewards from notebook 02 or a saved file
# For now we show the PPO eval curve only
if eval_log:
    steps, means, stds = zip(*eval_log)
    means, stds = np.array(means), np.array(stds)
    plt.figure(figsize=(10, 5))
    plt.plot(steps, means, marker="o", label="PPO — Deadly Corridor")
    plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
    plt.xlabel("Environment Steps")
    plt.ylabel("Eval Reward")
    plt.title("Sample Efficiency — PPO")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Save checkpoints

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)

# Full checkpoints with metadata for resumable training
for name, ag, wt, steps in [
    ("ppo_deadly_corridor", agent, wall_time_dc, total_env_steps),
    ("ppo_defend_the_center", agent_dtc, wall_time_dtc, dtc_total_steps),
]:
    checkpoint = {
        "model_state_dict": ag.network.state_dict(),
        "config": config,
        "total_env_steps": steps,
        "wall_time_seconds": wt,
    }
    torch.save(checkpoint, f"../checkpoints/{name}_full.pt")
    ag.save(f"../checkpoints/{name}.pt")

print("Checkpoints saved (full + weights-only) for both scenarios.")